# PlussII CodeM CUDA hardware check

Keep this notebook **private**. In Kaggle notebook settings, select an NVIDIA GPU and leave Internet on for the public GitHub checkout and Catch2. This notebook uses only the public PlussDev commit `d8c9e4e6030eb6635eec9cf7e72d15bd62abbddf`; it does not upload local working-tree changes. Build files and logs stay in `/kaggle/temp`, outside saved notebook output.

In [ ]:
from pathlib import Path
import os, re, shutil, subprocess, sys

def run(*args, cwd=None, capture=False):
    result = subprocess.run(args, cwd=cwd, check=True, text=True,
                            stdout=subprocess.PIPE if capture else None)
    return result.stdout.strip() if capture else None

for tool in ('git', 'nvidia-smi', 'nvcc'):
    if not shutil.which(tool):
        raise RuntimeError(f'{tool} is missing. Select a Kaggle NVIDIA GPU; if nvcc is still absent, this image cannot compile the CUDA kernels.')
run('nvidia-smi', '-L')
run('nvcc', '--version')
run(sys.executable, '-m', 'pip', 'install', '--quiet', 'cmake>=3.25,<4')
if not shutil.which('cmake'):
    raise RuntimeError('CMake was installed but is not on PATH')
run('cmake', '--version')


In [ ]:
commit = 'd8c9e4e6030eb6635eec9cf7e72d15bd62abbddf'
repo = Path('/kaggle/temp/PlussDev')
if not (repo / '.git').exists():
    run('git', 'clone', '--filter=blob:none', '--depth=1', '--sparse',
        'https://github.com/asanteyaw/PlussDev.git', str(repo))
run('git', 'sparse-checkout', 'set', 'PlussQit/src', 'PlussQit/include',
    'PlussQit/tests', 'PlussQit/tools', 'PlussMa/include', 'PlussTils/include', cwd=repo)
if run('git', 'rev-parse', 'HEAD', cwd=repo, capture=True) != commit:
    run('git', 'fetch', '--depth=1', 'origin', commit, cwd=repo)
    run('git', 'checkout', '--detach', 'FETCH_HEAD', cwd=repo)
actual = run('git', 'rev-parse', 'HEAD', cwd=repo, capture=True)
assert actual == commit, (actual, commit)
print('Testing public commit', actual)


In [ ]:
build = Path('/kaggle/temp/plussii-codem-cuda-build')
run('cmake', '-S', str(repo / 'PlussQit'), '-B', str(build),
    '-DCMAKE_BUILD_TYPE=Release', '-DCMAKE_CUDA_ARCHITECTURES=native',
    '-DPLUSSQIT_BUILD_CODEM=ON', '-DPLUSSQIT_BUILD_CODEH=OFF',
    '-DPLUSSQIT_BUILD_EXAMPLES=OFF', '-DPLUSSQIT_BUILD_TESTS=ON',
    '-DPLUSSQIT_ENABLE_CUDA=ON', '-DPLUSSQIT_FETCH_DEPS=ON',
    '-DPLUSSQIT_USE_FMT=OFF', '-DPLUSS_ENABLE_TORCH=OFF')
cache = (build / 'CMakeCache.txt').read_text()
if 'PLUSSQIT_ENABLE_CUDA:BOOL=ON' not in cache or 'CMAKE_CUDA_COMPILER:FILEPATH=' not in cache:
    raise RuntimeError('CUDA was not enabled in the CMake configuration')
run('cmake', '--build', str(build), '--target',
    'plussqit_codem_cuda_gate_tests', '--parallel', '2')


In [ ]:
gate = build / 'plussqit_codem_cuda_gate_tests'
assert gate.is_file(), gate
run(str(gate), '[cuda][bootstrap][inventory]', '--reporter', 'compact')
run(str(gate), '[cuda][providers]', '--reporter', 'compact')
run(str(gate), '[cuda][no_bridge]', '--reporter', 'compact')
print('Initial CUDA inventory, provider, and no-bridge gates passed.')
